# 2025 DL Lab4: Semantic Segmentation on BCSS

Before we start, please put **your name** and **SID** in following format: <br>
Hi I'm 陸仁賈, 314831000.

**Your Answer:**    
Hi I'm 戴良育, 314834012.

## Overview

- Semantic segmentation is a computer vision task that aims to classify each pixel in an image into specific objects or regions.

- In this assignment, you will implement a segmentor to classify the specific types of breast cancer lesions.

- The segmentor implemented in this assignment is U-Net, and you are required to construct it from scratch


## Kaggle Competition
Kaggle is an online community of data scientists and machine learning practitioners. Kaggle allows users to find and publish datasets, explore and build models in a web-based data-science environment, work with other data scientists and machine learning engineers, and enter competitions to solve data science challenges.

This assignment use kaggle to calculate your grade.  
Please use this [**LINK**](https://www.kaggle.com/t/250e9c1faedd423faa0eb2b1fe22b733) to join the competition.

## Unzip Data

Unzip BCSS.zip

+ `train` : Contains all training images
+ `val` : Contains all validation images
+ `test` : Contains all test images
+ `train_mask` : Contains all masks for the training set
+ `val_mask` : Contains all masks for the validation set

The train set contains 26,760 images, the val set contains 5,429 images, and the test set contains 4,000 images.

# Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
import torchvision
import torch.nn.functional as F
from torch.autograd import Variable

from PIL import Image
import cv2
import albumentations as A

import time
import os
from tqdm.notebook import tqdm

# 設置使用GPU 1和2
os.environ["CUDA_VISIBLE_DEVICES"] = "5,6"

# 檢查可用的GPU數量
if torch.cuda.is_available():
    print(f"可用的GPU數量: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    print("使用CPU訓練")

print(f"主要設備: {device}")

# Preprocessing

#### Verify that the paths to the image and mask are correct, and plot the position of the mask on the image.

First, the code establishes DataFrames for the paths of training and validation images, including their corresponding mask paths.

Second, it presents the total number of images in both the training and validation sets.

Third, the code loads a sample training image and its associated mask, visually demonstrating the image and the applied mask in a graphical representation.

In [ ]:
TRAIN_IMAGE_PATH = './BCSS/train/'
VAL_IMAGE_PATH = './BCSS/val/'
TRAIN_MASK_PATH = './BCSS/train_mask/'
VAL_MASK_PATH = './BCSS/val_mask/'

In [ ]:
n_classes = 3

def create_df(IMAGE_PATH):
    name = []
    for dirname, _, filenames in os.walk(IMAGE_PATH):
        for filename in filenames:
            name.append(filename.split('.')[0])

    return pd.DataFrame({'id': name}, index = np.arange(0, len(name)))

train_df = create_df(TRAIN_IMAGE_PATH)
val_df = create_df(VAL_IMAGE_PATH)


print('Total Train Images: ', len(train_df))
print('Total Val Images: ', len(val_df))

In [ ]:
X_train = train_df['id'].to_numpy()
X_val = val_df['id'].to_numpy()

In [ ]:
img = Image.open(TRAIN_IMAGE_PATH + train_df['id'][100] + '.png')
mask = Image.open(TRAIN_MASK_PATH + train_df['id'][100] + '.png')
print('Image Size', np.asarray(img).shape)
print('Mask Size', np.asarray(mask).shape)


plt.imshow(img)
plt.imshow(mask, alpha=0.6)
plt.title('Picture with Mask Appplied')
plt.show()

# Loading the Dataset

Define the BCSSDataset for loading the dataset, where each sample comprises an image and its corresponding mask

Build a classs inherit `torch.utils.data.Dataset`.
  
Implement `__init__`, `__getitem__` and `__len__` 3 functions.  

Some operations could be there: setting location of dataset, the method of reading data, label of dataset or transform of dataset.

See [torch.utils.data.Dataset](https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset) for more details

In [ ]:
class BCSSDataset(Dataset):

    def __init__(self, img_path, mask_path, X, mean, std, transform=None):
        self.img_path = img_path
        self.mask_path = mask_path
        self.X = X
        self.transform = transform
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        img = cv2.imread(self.img_path + self.X[idx] + '.png')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_path + self.X[idx] + '.png', cv2.IMREAD_GRAYSCALE)

        # 如果有變換（使用 albumentations）
        if self.transform is not None:
            # albumentations 需要使用命名參數
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']
            # ToTensorV2() 已經將 mask 轉換為 tensor，直接轉換為 long 類型
            mask = mask.long()
        else:
            # 如果沒有變換，手動進行標準化（使用 torchvision）
            t = T.Compose([T.ToTensor(), T.Normalize(self.mean, self.std)])
            img = t(img)
            # mask 轉換為 tensor
            mask = torch.from_numpy(mask).long()

        return img, mask

## Data augmentation

Data augmentation are techniques used to increase the amount of data by adding slightly modified copies of already existing data or newly created synthetic data from existing data.

PyTorch use `torchvision.transforms` to do data augmentation.
[You can see all function here.](https://pytorch.org/vision/stable/transforms.html)

In [ ]:
# import torchvision.transforms as v2
import albumentations as A
from albumentations.pytorch import ToTensorV2
mean=[0.485, 0.456, 0.406]
std=[0.229, 0.224, 0.225]

# For TRAIN
########################################################################
#  TODO: use transforms.xxx method to do some data augmentation        #
#  This one is for training, find the composition to get better result #
########################################################################
# transforms_train = v2.Compose([
#     v2.RandomHorizontalFlip(p=0.5),
#     v2.RandomVerticalFlip(p=0.5),
#     v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), 
#     v2.ToTensor(),
#     v2.Normalize(mean=mean, std=std)
# ])

transforms_train = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.GaussianBlur(blur_limit=(3, 7), p=0.5),
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

########################################################################
#                           End of your code                           #
########################################################################


# For VAL
########################################################################
#  For validation/test we avoid strong augmentations that change labels #
########################################################################
# transforms_val = v2.Compose([
#     v2.ToTensor(),
#     v2.Normalize(mean=mean, std=std)
# ])

transforms_val = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])
########################################################################
#                           End of your code                           #
########################################################################


#datasets
train_set = BCSSDataset(TRAIN_IMAGE_PATH, TRAIN_MASK_PATH, X_train, mean, std, transforms_train)
val_set = BCSSDataset(VAL_IMAGE_PATH, VAL_MASK_PATH, X_val, mean, std, transforms_val)

#dataloader
batch_size = 16

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=1, shuffle=False)

In [ ]:
train_set[0]

# U-net

U-net is a fully convolution neural network for image semantic segmentation. Consist of encoder and decoder parts connected with skip connections. Encoder extract features of different spatial resolution (skip connections) which are used by decoder to define accurate segmentation mask. Use concatenation for fusing decoder blocks with skip connections.


<!-- ![image](https://hackmd.io/_uploads/rJXtsY_Up.png) -->

In [ ]:
# class ConvBlock(nn.Module):

#     def __init__(self, in_channels, out_channels):
#         super(ConvBlock, self).__init__()

#         # number of input channels is a number of filters in the previous layer
#         # number of output channels is a number of filters in the current layer
#         # "same" convolutions
#         self.conv = nn.Sequential(
#             nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=True),
#             nn.BatchNorm2d(out_channels),
#             nn.ReLU(inplace=True),
#             nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=True),
#             nn.BatchNorm2d(out_channels),
#             nn.ReLU(inplace=True)
#         )

#     def forward(self, x):
#         x = self.conv(x)
#         return x


# class UpConv(nn.Module):

#     def __init__(self, in_channels, out_channels):
#         super(UpConv, self).__init__()

#         self.up = nn.Sequential(
#             nn.Upsample(scale_factor=2),
#             nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=True),
#             nn.BatchNorm2d(out_channels),
#             nn.ReLU(inplace=True)
#         )

#     def forward(self, x):
#         x = self.up(x)
#         return x


# class AttentionBlock(nn.Module):
#     """Attention block with learnable parameters"""

#     def __init__(self, F_g, F_l, n_coefficients):
#         """
#         :param F_g: number of feature maps (channels) in previous layer
#         :param F_l: number of feature maps in corresponding encoder layer, transferred via skip connection
#         :param n_coefficients: number of learnable multi-dimensional attention coefficients
#         """
#         super(AttentionBlock, self).__init__()

#         self.W_gate = nn.Sequential(
#             nn.Conv2d(F_g, n_coefficients, kernel_size=1, stride=1, padding=0, bias=True),
#             nn.BatchNorm2d(n_coefficients)
#         )

#         self.W_x = nn.Sequential(
#             nn.Conv2d(F_l, n_coefficients, kernel_size=1, stride=1, padding=0, bias=True),
#             nn.BatchNorm2d(n_coefficients)
#         )

#         self.psi = nn.Sequential(
#             nn.Conv2d(n_coefficients, 1, kernel_size=1, stride=1, padding=0, bias=True),
#             nn.BatchNorm2d(1),
#             nn.Sigmoid()
#         )

#         self.relu = nn.ReLU(inplace=True)

#     def forward(self, gate, skip_connection):
#         """
#         :param gate: gating signal from previous layer
#         :param skip_connection: activation from corresponding encoder layer
#         :return: output activations
#         """
#         g1 = self.W_gate(gate)
#         x1 = self.W_x(skip_connection)
#         psi = self.relu(g1 + x1)
#         psi = self.psi(psi)
#         out = skip_connection * psi
#         return out


In [ ]:
# class AttentionUNet(nn.Module):

#     def __init__(self, img_ch=3, output_ch=3):
#         super(AttentionUNet, self).__init__()

#         self.MaxPool = nn.MaxPool2d(kernel_size=2, stride=2)

#         # 編碼器路徑
#         self.Conv1 = ConvBlock(img_ch, 64)
#         self.Conv2 = ConvBlock(64, 128)
#         self.Conv3 = ConvBlock(128, 256)
#         self.Conv4 = ConvBlock(256, 512)
#         self.Conv5 = ConvBlock(512, 1024)

#         # 解碼器路徑
#         self.Up5 = UpConv(1024, 512)
#         self.Att5 = AttentionBlock(F_g=512, F_l=512, n_coefficients=256)
#         self.UpConv5 = ConvBlock(1024, 512)

#         self.Up4 = UpConv(512, 256)
#         self.Att4 = AttentionBlock(F_g=256, F_l=256, n_coefficients=128)
#         self.UpConv4 = ConvBlock(512, 256)

#         self.Up3 = UpConv(256, 128)
#         self.Att3 = AttentionBlock(F_g=128, F_l=128, n_coefficients=64)
#         self.UpConv3 = ConvBlock(256, 128)

#         self.Up2 = UpConv(128, 64)
#         self.Att2 = AttentionBlock(F_g=64, F_l=64, n_coefficients=32)
#         self.UpConv2 = ConvBlock(128, 64)

#         # 最終輸出層
#         self.Conv = nn.Conv2d(64, output_ch, kernel_size=1, stride=1, padding=0)

#     def forward(self, x):
#         """
#         前向傳播函數
#         e : 編碼器層
#         d : 解碼器層  
#         s : 從編碼器到解碼器的跳躍連接
#         """
#         # 編碼器路徑
#         e1 = self.Conv1(x)

#         e2 = self.MaxPool(e1)
#         e2 = self.Conv2(e2)

#         e3 = self.MaxPool(e2)
#         e3 = self.Conv3(e3)

#         e4 = self.MaxPool(e3)
#         e4 = self.Conv4(e4)

#         e5 = self.MaxPool(e4)
#         e5 = self.Conv5(e5)

#         # 解碼器路徑
#         d5 = self.Up5(e5)

#         # 注意力機制加權的跳躍連接
#         s4 = self.Att5(gate=d5, skip_connection=e4)
#         d5 = torch.cat((s4, d5), dim=1) # 將注意力加權的跳躍連接與前一層輸出拼接
#         d5 = self.UpConv5(d5)

#         d4 = self.Up4(d5)
#         s3 = self.Att4(gate=d4, skip_connection=e3)
#         d4 = torch.cat((s3, d4), dim=1)
#         d4 = self.UpConv4(d4)

#         d3 = self.Up3(d4)
#         s2 = self.Att3(gate=d3, skip_connection=e2)
#         d3 = torch.cat((s2, d3), dim=1)
#         d3 = self.UpConv3(d3)

#         d2 = self.Up2(d3)
#         s1 = self.Att2(gate=d2, skip_connection=e1)
#         d2 = torch.cat((s1, d2), dim=1)
#         d2 = self.UpConv2(d2)

#         # 最終輸出
#         out = self.Conv(d2)

#         return out

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConvolutionBlock(nn.Module):
    """
    A double convolution block with two sequential convolution layers.
    """
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class DownSample(nn.Module):
    """
    Down-sampling block with max pooling followed by a double convolution.
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.down = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConvolutionBlock(in_channels, out_channels)
        )

    def forward(self, x):
        return self.down(x)

class UpSample(nn.Module):
    """
    Up-sampling block with a up-sampling layer followed by a double convolution.
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels , in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConvolutionBlock(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class OutputConvolution(nn.Module):
    """
    The final convolution layer to map the features to the desired number of classes.
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    """
    U-Net architecture for semantic segmentation.
    """
    def __init__(self, num_channels, num_classes):
        super().__init__()
        self.inc = DoubleConvolutionBlock(num_channels, 64)
        self.down1 = DownSample(64, 128)
        self.down2 = DownSample(128, 256)
        self.down3 = DownSample(256, 512)
        self.down4 = DownSample(512, 1024)
        self.up1 = UpSample(1024, 512)
        self.up2 = UpSample(512, 256)
        self.up3 = UpSample(256, 128)
        self.up4 = UpSample(128, 64)
        self.outc = OutputConvolution(64, num_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits

In [ ]:
# 初始化模型
model = UNet(3, 3)

# 如果有多個GPU，使用 DataParallel 進行多GPU訓練
if torch.cuda.device_count() > 1:
    print(f"使用 {torch.cuda.device_count()} 個GPU進行訓練")
    model = nn.DataParallel(model)
    
model = model.to(device)
print(f"模型已移動到: {device}")

In [ ]:
model

# Training

In this section, you will implement some functions in your training loop. There are several crucial functions you need to implement:

- Pixel Accuracy: Pixel Accuracy measures the percentage of correctly predicted pixels out of the total pixels in the image

- mIoU (Mean Intersection over Union): mIoU evaluates the spatial overlap between the predicted and ground truth segmentation masks for multiple classes.

- Dice Loss: Dice Loss quantifies the dissimilarity between the predicted and ground truth masks, emphasizing the agreement between the two masks

Try to use labeled data design and train a segmentor (Unet) from scratch to predict the mask of a breast cancer lesions image.

In [ ]:
def pixel_accuracy(output, mask):
    with torch.no_grad():
        #############################################################################################################
        #  1. Convert the output to class predictions using argmax after applying softmax                           #
        #  2. Create a tensor of binary values indicating correct predictions                                       #
        #  3. Calculate accuracy by dividing the number of correct predictions by the total number of predictions   #
        #############################################################################################################
        output = torch.argmax(F.softmax(output, dim=1), dim=1)
        correct = (output == mask)
        accuracy = correct.sum() / correct.numel()
        accuracy = accuracy.item()
        #############################################################################################################
        #                                      End of your code                                                     #
        #############################################################################################################
    return accuracy

## mIoU

mean Intersection over Union (mIoU), a metric used to evaluate the performance of a segmentation model. It takes predicted masks and ground truth masks as input, computes the IoU for each class, and returns the average IoU across all classes.

![image](https://hackmd.io/_uploads/SkLIbhFL6.png)

In [ ]:
def mIoU(pred_mask, mask, n_classes=3, ignore_class=0):
    """
    For batch input, calculates per-image mIoU and averages across the batch.
    """
    with torch.no_grad():
        probs = F.softmax(pred_mask, dim=1)
        preds = torch.argmax(probs, dim=1)  # (B, H, W)

        batch_size = preds.shape[0]
        batch_miou_scores = []

        # Calculate mIoU for each image in the batch (per-image averaging)
        for b in range(batch_size):
            pred_img = preds[b]  # (H, W)
            mask_img = mask[b]   # (H, W)

            iou_list = []
            
            for cls in range(n_classes):
                pred_c = (pred_img == cls)
                label_c = (mask_img == cls)

                intersection = (pred_c & label_c).sum().float()
                union = (pred_c | label_c).sum().float()

                # Skip if union is 0 (class doesn't exist in both GT and Pred)
                if union == 0:
                    continue

                iou = intersection / union

                if cls != ignore_class:
                    iou_list.append(iou)

            # Calculate mIoU for this image
            if len(iou_list) > 0:
                img_miou = torch.stack(iou_list).mean()
                batch_miou_scores.append(img_miou)

        # Return mean of per-image mIoU scores
        if len(batch_miou_scores) > 0:
            return float(torch.stack(batch_miou_scores).mean().item())
        else:
            return 0.0

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss for semantic segmentation tasks.
    
    Loss(x, class) = - alpha * (1 - softmax(x)[class])^gamma * log(softmax(x)[class])
    
    Args:
        alpha: 類別權重係數，預設為 1.0
        gamma: 聚焦參數，預設為 2.0。gamma 越大，對簡單樣本的懲罰越小
        reduction: 'mean' 或 'sum'
    """
    def __init__(self, alpha=1.0, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        """
        Args:
            inputs: 模型輸出 (B, C, H, W) - logits
            targets: 真實標籤 (B, H, W) - 類別索引
        """
        # 計算 log softmax
        log_probs = F.log_softmax(inputs, dim=1)  # (B, C, H, W)
        
        # 獲取每個像素對應類別的 log probability
        # targets: (B, H, W) -> (B, 1, H, W)
        targets_unsqueezed = targets.unsqueeze(1)
        
        # 使用 gather 提取對應類別的 log probability
        # log_pt: (B, 1, H, W)
        log_pt = log_probs.gather(1, targets_unsqueezed)
        
        # 移除多餘的維度: (B, 1, H, W) -> (B, H, W)
        log_pt = log_pt.squeeze(1)
        
        # 計算 pt (probability)
        pt = torch.exp(log_pt)
        
        # 計算 focal weight: (1 - pt)^gamma
        focal_weight = (1 - pt) ** self.gamma
        
        # 計算 focal loss
        focal_loss = -self.alpha * focal_weight * log_pt
        
        # 應用 reduction
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

## Dice Loss

Dice loss is based on Sørensen-Dice coefficient. It measures the overlap between the predicted and target segmentation masks. Dice loss provides a differentiable and smooth measure of segmentation accuracy.

${DiceLoss}(y, \bar p) = 1 - \cfrac{(2y\bar p + \epsilon)}{(y + \bar p + \epsilon)}$

- $y$ represents the ground truth mask.
- $\bar p$ represents the predicted mask.
- $\epsilon$ is a small constant added for numerical stability.

In [ ]:
class DiceLoss(nn.Module):
    """
    Dice loss
    """

    def __init__(self):
        super(DiceLoss, self).__init__()

    def forward(self, inputs, targets, eps=1e-6):
        """
        Calculation of dice loss

        :param inputs: model predictions
        :param targets: target values
        :param eps: stability factor, defaults to 1e-6
        :return: loss value
        """
        #
        #######################################
        #        implement dice loss          #
        #######################################
        # 對輸入應用 softmax 得到概率分佈
        inputs = F.softmax(inputs, dim=1)
        
        # 將目標標籤轉換為 one-hot 編碼
        num_classes = inputs.size(1)
        targets_one_hot = F.one_hot(targets, num_classes=num_classes).permute(0, 3, 1, 2).float()
        
        # 將張量展平以便計算
        inputs = inputs.contiguous().view(inputs.size(0), inputs.size(1), -1)
        targets_one_hot = targets_one_hot.contiguous().view(targets_one_hot.size(0), targets_one_hot.size(1), -1)
        
        # 計算每個類別的交集和聯集
        intersection = (inputs * targets_one_hot).sum(dim=2)
        union = inputs.sum(dim=2) + targets_one_hot.sum(dim=2)
        
        # 計算每個類別的 dice 係數
        dice = (2.0 * intersection + eps) / (union + eps)
        #######################################
        #          End of your code           #
        #######################################
        return 1.0 - dice.mean()

# Training loop

Call train function in a loop.  
Take a break and wait.

In [ ]:
def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group['lr']

def fit(epochs, model, train_loader, val_loader, criterion1, criterion2, optimizer, scheduler, patch=False):
    torch.cuda.empty_cache()
    train_losses = []
    test_losses = []
    val_iou = []; val_acc = []
    train_iou = []; train_acc = []
    lrs = []
    min_loss = np.inf
    best_iou = 0.0  # 追蹤最佳 mIoU
    not_improve = 0  # 連續未改善的次數

    model.to(device)
    fit_time = time.time()
    for e in range(epochs):
        since = time.time()
        running_loss = 0
        iou_score = 0
        accuracy = 0
        #training loop
        model.train()
        for i, data in enumerate(tqdm(train_loader)):
            #training phase
            ########################################################################
            # TODO: Forward, backward and optimize                                 #
            # 1. process input through the network                                 #
            # 2. compute the two losses                                            #
            # 3. caluate mIoU and piexl accuracy                                   #
            # 4. propagate gradients back into the network's parameters            #
            # 5. update the weights of the network                                 #
            # 6. reset gradient                                                    #
            ########################################################################
            image, mask = data

            # 將數據移到指定設備
            image = image.to(device)
            mask = mask.to(device)
            
            # 前向傳播
            output = model(image)
            
            # 計算損失（FocalLoss + Dice）
            loss = criterion1(output, mask) + criterion2(output, mask)
            
            # 反向傳播
            loss.backward()
            
             # 更新權重
            optimizer.step()
            
            # 重置梯度
            optimizer.zero_grad()
            
            # 計算指標
            iou_score += mIoU(output, mask)
            accuracy += pixel_accuracy(output, mask)
            
            ########################################################################
            #                           End of your code                           #
            ########################################################################
            #step the learning rate
            lrs.append(get_lr(optimizer))
            scheduler.step()

            running_loss += loss.item()

        else:
            model.eval()
            test_loss = 0
            test_accuracy = 0
            val_iou_score = 0
            #validation loop
            with torch.no_grad():
                for i, data in enumerate(tqdm(val_loader)):
                    ########################################################################
                    # TODO: Forward, backward and optimize                                 #
                    # 1. process input through the network                                 #
                    # 2. compute the two losses                                            #
                    # 3. caluate mIoU and piexl accuracy                                   #
                    # 4. save this batch's loss into test_loss                             #
                    ########################################################################
                    image, mask = data

                    # 將數據移到指定設備
                    image = image.to(device)
                    mask = mask.to(device)
                    
                    # 前向傳播（驗證時不需要梯度）
                    output = model(image)
                    
                    # 計算指標
                    val_iou_score += mIoU(output, mask)
                    test_accuracy += pixel_accuracy(output, mask)
                    # 計算損失
                    loss = criterion1(output, mask) + criterion2(output, mask)
                    test_loss += loss.item()
                    ########################################################################
                    #                           End of your code                           #
                    ########################################################################

            #calculatio mean for each batch
            train_losses.append(running_loss/len(train_loader))
            test_losses.append(test_loss/len(val_loader))

            # 計算當前 epoch 的平均驗證損失和 mIoU
            current_val_loss = test_loss/len(val_loader)
            current_val_iou = val_iou_score/len(val_loader)
            
            # 檢查是否有改善（基於 loss）
            if current_val_loss < min_loss:
                print('Loss Decreasing.. {:.3f} >> {:.3f} '.format(min_loss, current_val_loss))
                min_loss = current_val_loss
                not_improve = 0  # 重置連續未改善計數器
                
                # 保存最佳 loss 模型
                print('Saving best loss model...')
                torch.save(model, 'Unet1_best_loss.pt')
            else:
                not_improve += 1  # 增加連續未改善次數
                print(f'Loss Not Decrease for {not_improve} consecutive time(s)')
                if not_improve == 7:
                    print('Loss not decrease for 7 consecutive times, Stop Training')
                    break
            
            # 檢查是否有最佳 mIoU（另外保存）
            if current_val_iou > best_iou:
                print('Best mIoU Improved: {:.3f} >> {:.3f}'.format(best_iou, current_val_iou))
                best_iou = current_val_iou
                print('Saving best mIoU model...')
                torch.save(model, 'Unet1_best_miou.pt')

            #iou
            val_iou.append(val_iou_score/len(val_loader))
            train_iou.append(iou_score/len(train_loader))
            train_acc.append(accuracy/len(train_loader))
            val_acc.append(test_accuracy/ len(val_loader))
            print("Epoch:{}/{}..".format(e+1, epochs),
                  "Train Loss: {:.3f}..".format(running_loss/len(train_loader)),
                  "Val Loss: {:.3f}..".format(test_loss/len(val_loader)),
                  "Train mIoU:{:.3f}..".format(iou_score/len(train_loader)),
                  "Val mIoU: {:.3f}..".format(val_iou_score/len(val_loader)),
                  "Train Acc:{:.3f}..".format(accuracy/len(train_loader)),
                  "Val Acc:{:.3f}..".format(test_accuracy/len(val_loader)),
                  "Time: {:.2f}m".format((time.time()-since)/60))

    history = {'train_loss' : train_losses, 'val_loss': test_losses,
               'train_miou' :train_iou, 'val_miou':val_iou,
               'train_acc' :train_acc, 'val_acc':val_acc,
               'lrs': lrs}
    print('Total time: {:.2f} m' .format((time.time()- fit_time)/60))
    return history

In [ ]:
################################################################################
#     You can adjust those hyper parameters to loop for max_epochs times       #
################################################################################
max_lr = 1e-3
epoch = 15
weight_decay = 1e-4
criterion1 = nn.CrossEntropyLoss()
criterion2 = DiceLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=weight_decay)
sched = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr, epochs=epoch, steps_per_epoch=len(train_loader))

history = fit(epoch, model, train_loader, val_loader, criterion1, criterion2, optimizer, sched)
################################################################################
#                               End of your code                               #
################################################################################

In [ ]:
torch.save(model, 'Unet-Resnet.pt')

#### Visualize accuracy and loss

In [ ]:
def plot_loss(history):
    plt.plot(history['val_loss'], label='val', marker='o')
    plt.plot( history['train_loss'], label='train', marker='o')
    plt.title('Loss per epoch'); plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend(), plt.grid()
    plt.show()

def plot_score(history):
    plt.plot(history['train_miou'], label='train_mIoU', marker='*')
    plt.plot(history['val_miou'], label='val_mIoU',  marker='*')
    plt.title('Score per epoch'); plt.ylabel('mean IoU')
    plt.xlabel('epoch')
    plt.legend(), plt.grid()
    plt.show()

def plot_acc(history):
    plt.plot(history['train_acc'], label='train_accuracy', marker='*')
    plt.plot(history['val_acc'], label='val_accuracy',  marker='*')
    plt.title('Accuracy per epoch'); plt.ylabel('Accuracy')
    plt.xlabel('epoch')
    plt.legend(), plt.grid()
    plt.show()

In [ ]:
plot_loss(history)
plot_score(history)
plot_acc(history)

### Predict Result

Predict the labesl based on testing set. Upload to [Kaggle](https://www.kaggle.com/t/250e9c1faedd423faa0eb2b1fe22b733).

**How to upload**

1. Click the folder icon in the left hand side of Colab.
2. Right click "result.csv". Select "Download"
3. To kaggle. Click "Submit Predictions"
4. Upload the result.csv
5. System will automaticlaly calculate the accuracy of 50% dataset and publish this result to leaderboard.

In [ ]:
TEST_IMAGE_PATH = './BCSS/test/'

In [ ]:
test_df = create_df(TEST_IMAGE_PATH)

In [ ]:
X_test = test_df['id'].to_numpy()

In [ ]:
class BCSSDataset(Dataset):
    def __init__(self, img_path, mask_path, X, mean, std):
        self.img_path = img_path
        self.mask_path = mask_path
        self.X = X
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        img = cv2.imread(self.img_path + self.X[idx] + '.png')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        t = T.Compose([T.ToTensor(), T.Normalize(self.mean, self.std)])
        img = t(img)
        return img, self.X[idx]
    
test_set = BCSSDataset(TEST_IMAGE_PATH, None, X_test, mean, std)

In [ ]:
# 載入完整的模型（不是 state_dict）
loaded_model = torch.load('Unet-Resnet.pt')

# 如果載入的模型是 DataParallel 包裝的，需要提取內部模型
if isinstance(loaded_model, nn.DataParallel):
    model = loaded_model.module
else:
    model = loaded_model

model = model.to(device)
model.eval()  # 設置為評估模式

In [ ]:
def predict_image(model, image):
    model.eval()
    model.to(device); image=image.to(device)
    with torch.no_grad():
        image = image.unsqueeze(0)
        output = model(image)
        masked = torch.argmax(output, dim=1)
        masked = masked.cpu().squeeze(0)
    return masked

In [ ]:
import pandas as pd
from tqdm import tqdm

data = []

for i in tqdm(range(len(test_set))):
    img, filename = test_set[i]
    pred_mask = predict_image(model, img)

    data.append({'index': filename, 'pred_mask': pred_mask.numpy().tolist()})

df = pd.DataFrame(data)

df.to_csv('output.csv', index=False)